<a target="_blank" href="https://colab.research.google.com/github/lukebarousse/Int_SQL_Data_Analytics_Course/blob/main/Resources/Blank_SQL_Notebook.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# PYTHON PROJECT Notebook

#### Import Libraries & Upload Data File for Analysis

In [ ]:
# =========================================================
# IMPORT LIBRARIES
# =========================================================
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px

# =========================================================
# PAGE CONFIGURATION (MUST BE FIRST STREAMLIT COMMAND)
# =========================================================
st.set_page_config(
    page_title="Data Analysis App",
    page_icon="📊",
    layout="wide"
)

# =========================================================
# APP HEADER
# =========================================================
st.title("📊 Interactive Data Analysis & Visualization App")
st.markdown(
    "Upload any **CSV or Excel file** to explore, analyze, and visualize your data."
)

# =========================================================
# FILE UPLOAD
# =========================================================
uploaded_file = st.file_uploader(
    "📂 Upload your dataset",
    type=["csv", "xlsx", "xls"]
)

# =========================================================
# DATA LOADING FUNCTION (CACHED FOR PERFORMANCE)
# =========================================================
@st.cache_data
def load_data(file):
    if file.name.lower().endswith(".csv"):
        return pd.read_csv(file)
    return pd.read_excel(file)

# =========================================================
# ALWAYS SHOW APP STATUS (PREVENT BLANK SCREEN CONFUSION)
# =========================================================
st.write("🚀 App is running...")

# =========================================================
# MAIN APPLICATION LOGIC
# =========================================================
if uploaded_file is not None:

    # -------------------------
    # LOAD DATA
    # -------------------------
    df = load_data(uploaded_file)
    st.success("✅ File uploaded and loaded successfully")

    # -------------------------
    # DATASET OVERVIEW
    # -------------------------
    st.subheader("🔍 Dataset Overview")

    col1, col2, col3 = st.columns(3)
    col1.metric("Rows", df.shape[0])
    col2.metric("Columns", df.shape[1])
    col3.metric("Missing Values", int(df.isna().sum().sum()))

    st.dataframe(df.head(), use_container_width=True)

    # -------------------------
    # SIDEBAR FILTERS
    # -------------------------
    st.sidebar.header("🔎 Filter Data")

    filtered_df = df.copy()

    cat_columns = filtered_df.select_dtypes(
        include=["object", "category"]
    ).columns

    for column in cat_columns:
        unique_values = sorted(
            filtered_df[column].dropna().astype(str).unique()
        )

        if len(unique_values) > 1:
            selected_values = st.sidebar.multiselect(
                f"Filter by {column}",
                options=unique_values,
                default=unique_values
            )

            filtered_df = filtered_df[
                filtered_df[column].astype(str).isin(selected_values)
            ]

    if filtered_df.empty:
        st.warning("⚠️ No data available after applying filters.")
        st.stop()

    # -------------------------
    # KPI SECTION
    # -------------------------
    st.subheader("📌 Key Performance Indicators (KPIs)")

    numeric_columns = filtered_df.select_dtypes(include=np.number).columns

    if len(numeric_columns) > 0:
        k1, k2, k3 = st.columns(3)

        k1.metric(
            label=f"Total {numeric_columns[0]}",
            value=f"{filtered_df[numeric_columns[0]].sum():,.2f}"
        )

        if len(numeric_columns) > 1:
            k2.metric(
                label=f"Average {numeric_columns[1]}",
                value=f"{filtered_df[numeric_columns[1]].mean():,.2f}"
            )

        if len(numeric_columns) > 2:
            k3.metric(
                label=f"Maximum {numeric_columns[2]}",
                value=f"{filtered_df[numeric_columns[2]].max():,.2f}"
            )
    else:
        st.info("No numeric columns available for KPI calculation.")

    # -------------------------
    # BAR CHART VISUALIZATION
    # -------------------------
    st.subheader("📊 Bar Chart Analysis")

    if len(cat_columns) > 0 and len(numeric_columns) > 0:

        selected_category = st.selectbox(
            "Select a categorical column",
            cat_columns
        )

        selected_metric = st.selectbox(
            "Select a numeric column",
            numeric_columns
        )

        bar_df = (
            filtered_df
            .groupby(selected_category)[selected_metric]
            .sum()
            .reset_index()
            .sort_values(by=selected_metric, ascending=False)
        )

        bar_fig = px.bar(
            bar_df,
            x=selected_category,
            y=selected_metric,
            title=f"{selected_metric} by {selected_category}"
        )

        st.plotly_chart(bar_fig, use_container_width=True)

    else:
        st.info("Bar chart requires at least one categorical and numeric column.")

    # -------------------------
    # DISTRIBUTION ANALYSIS
    # -------------------------
    st.subheader("📈 Distribution Analysis")

    if len(numeric_columns) > 0:
        dist_col = st.selectbox(
            "Select numeric column for distribution",
            numeric_columns
        )

        hist_fig = px.histogram(
            filtered_df,
            x=dist_col,
            nbins=30,
            title=f"Distribution of {dist_col}"
        )

        st.plotly_chart(hist_fig, use_container_width=True)

else:
    st.info("📥 Upload a CSV or Excel file to start your analysis.")

# =========================================================
# CUSTOM STYLING
# =========================================================
st.markdown(
    """
    <style>
    div[data-testid="metric-container"] {
        background-color: #f4f6f9;
        padding: 15px;
        border-radius: 10px;
        border-left: 5px solid #4f8bf9;
    }
    </style>
    """,
    unsafe_allow_html=True
)



In [ ]:
%%sql

